### Источники данных:
* Данные по **ключевой ставке** были взяты из: [cbr.ru/hd_base/keyrate/](https://www.cbr.ru/hd_base/keyrate/)
* Данные по **Ииндексу потребительских цен** были взяты из: [rosstat.gov.ru/statistics/price](https://rosstat.gov.ru/statistics/price)

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# ── 1. Данные по мероприятиям ──────────────────────────────
df_events = pd.read_excel('../data/Лофты посвяты.xlsx')
df_events.columns = ['Дата', 'Тип', 'Минимальная цена', 'Средняя цена', 'Ссылки']
df_events['Дата'] = pd.to_datetime(df_events['Дата'], dayfirst=True, errors='coerce')
df_events['Год'] = df_events['Дата'].dt.year
for col in ['Минимальная цена', 'Средняя цена']:
    df_events[col] = df_events[col].astype(str).str.replace(',', '.', regex=False)
    df_events[col] = df_events[col].str.replace(r'[^\d\.\-]', '', regex=True)
    df_events[col] = pd.to_numeric(df_events[col], errors='coerce')

df_events_clean = df_events.dropna(subset=['Год', 'Средняя цена']).copy()
df_events_clean['Тип'] = df_events_clean['Тип'].astype(str).str.strip().str.lower()

loft    = df_events_clean[df_events_clean['Тип'] == 'лофт']
posvyat = df_events_clean[df_events_clean['Тип'] == 'посвят']

loft_yearly    = loft.groupby('Год')['Средняя цена'].mean().reset_index()
loft_yearly['Тип'] = 'Лофт'
posvyat_yearly = posvyat.groupby('Год')['Средняя цена'].mean().reset_index()
posvyat_yearly['Тип'] = 'Посвят'

# ── 2. ИПЦ ────────────────────────────────────────────────
df_ipc = pd.read_excel('../data/ИПЦ.xlsx')
df_ipc.columns = ['Год', 'ИПЦ']
df_ipc_yearly = df_ipc.dropna().copy()
df_ipc_yearly['Год'] = df_ipc_yearly['Год'].astype(int)
df_ipc_yearly = df_ipc_yearly.sort_values('Год').reset_index(drop=True)

# ── 3. Макро-данные (ИПЦ по категориям + ключевая ставка) ──
df_macro = pd.read_csv('../data/macro.csv')
for col in df_macro.columns:
    if col != 'Год':
        df_macro[col] = (df_macro[col].astype(str)
                         .str.replace(',', '.', regex=False)
                         .str.replace('%', '', regex=False)
                         .str.strip())
        df_macro[col] = pd.to_numeric(df_macro[col], errors='coerce')
df_macro = df_macro.rename(columns={
    'ИПЦ общий (%)':         'ИПЦ общий',
    'ИПЦ Алкоголь (%)':      'ИПЦ алкоголь/табак',
    'ИПЦ Культура (%)':      'ИПЦ культура/досуг',
    'ИПЦ Аренда (%)':        'ИПЦ аренда жилья',
    'Средняя ставка ЦБ (%)': 'Ключевая ставка',
})
df_macro['Реальные доходы'] = np.nan  # данные не включены в локальный источник

# ── 4. Сборка df_main ──────────────────────────────────────
loft_yr = loft_yearly[['Год', 'Средняя цена']].rename(columns={'Средняя цена': 'Цена лофт'})
posv_yr = posvyat_yearly[['Год', 'Средняя цена']].rename(columns={'Средняя цена': 'Цена посвят'})

year_min = int(min(loft_yr['Год'].min(), posv_yr['Год'].min()))
year_max = int(max(loft_yr['Год'].max(), posv_yr['Год'].max()))
all_years = pd.DataFrame({'Год': range(year_min, year_max + 1)})

df_main = (
    all_years
    .merge(loft_yr,  on='Год', how='left')
    .merge(posv_yr,  on='Год', how='left')
    .merge(df_macro, on='Год', how='left')
)

print('✅ Данные загружены из папки data')
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    display(df_main)
print(f"\nПериод: {df_main['Год'].min()} — {df_main['Год'].max()}")

## Блок 2. Тренды и сравнение периодов

Сначала оцениваем общий линейный тренд:

$$
P_t = \alpha + \beta \cdot t + \varepsilon_t
$$

где:
- $P_t$ — цена события  
- $t$ — год  
- $\beta$ — средний годовой рост цены  

Затем делим ряд на два периода:
- **до 2023 года**
- **с 2023 года**

и оцениваем отдельные тренды:

$$
P_t = \alpha_1 + \beta_1 \cdot t + \varepsilon_t \quad (t < 2023)
$$

$$
P_t = \alpha_2 + \beta_2 \cdot t + \varepsilon_t \quad (t \ge 2023)
$$

Сравнение $\beta_1$ и $\beta_2$ позволяет проверить гипотезу о «точке перегиба» и изменении динамики цен.

In [ ]:
from scipy import stats
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error
import warnings

warnings.filterwarnings("ignore")

print("=" * 60)
print("ТРЕНД ЦЕН (регрессия цена ~ год)")
print("H₀: тренда нет | H₁: цены меняются вместе со временем")
print("=" * 60)

fig = go.Figure()

for col, label, color in [('Цена лофт', 'Лофты', 'fuchsia'), ('Цена посвят', 'Посвяты', 'teal')]:
    sub = df_main[['Год', col]].dropna()

    if len(sub) >= 2:
        slope, intercept, r, p, _ = stats.linregress(sub['Год'], sub[col])
        trend = slope * sub['Год'] + intercept
        verdict = "✅ тренд значим" if p < 0.05 else "❌ тренд не значим"

        print(f"\n{label}:")
        print(f"  Рост: {slope:+,.0f} ₽ в год")
        print(f"  R² = {r**2:.3f}")
        print(f"  p-value = {p:.4f}")
        print(f"  → {verdict}")

        fig.add_trace(go.Scatter(x=sub['Год'], y=sub[col], mode='lines+markers',
                                 name=label, line=dict(color=color, width=3), marker=dict(size=10)))
        fig.add_trace(go.Scatter(x=sub['Год'], y=trend, mode='lines',
                                 name=f'Тренд ({label})', line=dict(color=color, dash='dash', width=2)))

fig.update_layout(title='Тренд цен: Лофты и Посвяты',
                  xaxis_title='Год', yaxis_title='Средняя цена, ₽',
                  template='plotly_white', hovermode='x unified',
                  xaxis=dict(tickmode='linear', dtick=1))
fig.show()

ТРЕНД ЦЕН (регрессия цена ~ год)
H₀: тренда нет | H₁: цены меняются вместе со временем

Лофты:
  Рост: +157 ₽ в год
  R² = 0.894
  p-value = 0.0004
  → ✅ тренд значим

Посвяты:
  Рост: +106 ₽ в год
  R² = 0.593
  p-value = 0.0152
  → ✅ тренд значим


## Блок 3. Стационарность рядов

Проверяем уровни и первые разности.

- **ADF**:  
  \(H_0\): ряд нестационарен  
- **KPSS**:  
  \(H_0\): ряд стационарен  

Если уровни нестационарны, а первые разности стационарны, для ARIMA берём \(d=1\).

In [ ]:
print("=" * 60)
print("ТЕСТ НА СТАЦИОНАРНОСТЬ")
print("ADF: H₀ — нестационарен | p < 0.05 → стационарен")
print("KPSS: H₀ — стационарен | p < 0.05 → нестационарен")
print("=" * 60)

for col, label in [('Цена лофт', 'Лофты'), ('Цена посвят', 'Посвяты')]:
    series = df_main[col].dropna()

    if len(series) >= 4:
        adf_stat, adf_p, _, _, _, _ = adfuller(series, autolag='AIC')
        kpss_stat, kpss_p, _, _ = kpss(series, regression='c', nlags='auto')

        print(f"\n{label} — уровни:")
        print(f"  ADF:  статистика = {adf_stat:.3f}, p = {adf_p:.4f}")
        print(f"  KPSS: статистика = {kpss_stat:.3f}, p = {kpss_p:.4f}")

        if len(series.diff().dropna()) >= 4:
            diff_series = series.diff().dropna()
            adf_diff_stat, adf_diff_p, _, _, _, _ = adfuller(diff_series, autolag='AIC')
            kpss_diff_stat, kpss_diff_p, _, _ = kpss(diff_series, regression='c', nlags='auto')

            print(f"  {label} — первые разности:")
            print(f"    ADF:  статистика = {adf_diff_stat:.3f}, p = {adf_diff_p:.4f}")
            print(f"    KPSS: статистика = {kpss_diff_stat:.3f}, p = {kpss_diff_p:.4f}")

            if adf_p >= 0.05 and kpss_p < 0.05 and adf_diff_p < 0.05 and kpss_diff_p >= 0.05:
                print("  → уровни нестационарны, первые разности стационарны: для ARIMA берём d=1")
            elif adf_p < 0.05 and kpss_p >= 0.05:
                print("  → ряд в уровнях выглядит стационарным")
            else:
                print("  → результат смешанный, интерпретируем осторожно")

ТЕСТ НА СТАЦИОНАРНОСТЬ
ADF: H₀ — нестационарен | p < 0.05 → стационарен
KPSS: H₀ — стационарен | p < 0.05 → нестационарен

Лофты — уровни:
  ADF:  статистика = 2.167, p = 0.9989
  KPSS: статистика = 0.464, p = 0.0497
  Лофты — первые разности:
    ADF:  статистика = -1.452, p = 0.5570
    KPSS: статистика = 0.394, p = 0.0798
  → результат смешанный, интерпретируем осторожно

Посвяты — уровни:
  ADF:  статистика = -2.308, p = 0.1695
  KPSS: статистика = 0.406, p = 0.0746
  Посвяты — первые разности:
    ADF:  статистика = -2.140, p = 0.2289
    KPSS: статистика = 0.385, p = 0.0837
  → результат смешанный, интерпретируем осторожно


In [ ]:
print("=" * 60)
print("СРАВНЕНИЕ ТРЕНДОВ: ДО И ПОСЛЕ 2023 ГОДА")
print("Смотрим, меняется ли наклон тренда после 2023 года")
print("=" * 60)

BREAK_YEAR = 2023

for col, label, color in [('Цена лофт', 'Лофты', 'fuchsia'), ('Цена посвят', 'Посвяты', 'teal')]:
    sub = df_main[['Год', col]].dropna()
    before = sub[sub['Год'] < BREAK_YEAR]
    after = sub[sub['Год'] >= BREAK_YEAR]

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=sub['Год'], y=sub[col], mode='lines+markers',
                             name=label, line=dict(color=color, width=3), marker=dict(size=10)))

    print(f"\n{label}:")
    if len(before) >= 2:
        slope_b, intercept_b, r_b, p_b, _ = stats.linregress(before['Год'], before[col])
        trend_b = slope_b * before['Год'] + intercept_b
        print(f"  До 2023:  slope = {slope_b:+,.0f} ₽/год, R² = {r_b**2:.3f}, p = {p_b:.4f}")
        fig.add_trace(go.Scatter(x=before['Год'], y=trend_b, mode='lines',
                                 name='Тренд до 2023', line=dict(color='orange', dash='dash', width=2)))
    else:
        slope_b = np.nan
        print("  До 2023: недостаточно точек для регрессии")

    if len(after) >= 2:
        slope_a, intercept_a, r_a, p_a, _ = stats.linregress(after['Год'], after[col])
        trend_a = slope_a * after['Год'] + intercept_a
        print(f"  С 2023:   slope = {slope_a:+,.0f} ₽/год, R² = {r_a**2:.3f}, p = {p_a:.4f}")
        fig.add_trace(go.Scatter(x=after['Год'], y=trend_a, mode='lines',
                                 name='Тренд с 2023', line=dict(color='red', dash='dot', width=2)))
    else:
        slope_a = np.nan
        print("  С 2023: недостаточно точек для регрессии")

    if pd.notna(slope_b) and pd.notna(slope_a):
        print(f"  Δ slope = {slope_a - slope_b:+,.0f} ₽/год")

    fig.add_vline(x=BREAK_YEAR, line_dash='dot', line_color='gray', line_width=2,
                  annotation_text='2023', annotation_position='top right')
    fig.update_layout(title=f'{label}: сравнение трендов до и после 2023 года',
                      xaxis_title='Год', yaxis_title='Средняя цена, ₽',
                      template='plotly_white', hovermode='x unified',
                      xaxis=dict(tickmode='linear', dtick=1))
    fig.show()

СРАВНЕНИЕ ТРЕНДОВ: ДО И ПОСЛЕ 2023 ГОДА
Смотрим, меняется ли наклон тренда после 2023 года

Лофты:
  До 2023:  slope = +70 ₽/год, R² = 0.864, p = 0.0705
  С 2023:   slope = +281 ₽/год, R² = 0.990, p = 0.0049
  Δ slope = +211 ₽/год



Посвяты:
  До 2023:  slope = +196 ₽/год, R² = 0.809, p = 0.0147
  С 2023:   slope = -108 ₽/год, R² = 0.345, p = 0.6001
  Δ slope = -304 ₽/год


## Блок 4. Макро-связи и прогноз

Чтобы не ловить ложные корреляции из-за общего тренда, связи с макропоказателями считаем на **темпах роста**, а не на уровнях:

$$
g_t = \left(\frac{P_t}{P_{t-1}} - 1\right)\cdot 100
$$

где:
- $g_t$ — темп роста цены  
- $P_t$ — цена в году $t$  

Далее анализируем связь:
- темпы роста лофтов vs ИПЦ  
- темпы роста посвятов vs ИПЦ  

В конце сравниваем несколько простых ARIMA-моделей (например, $(1,1,0)$, $(0,1,1)$, $(1,1,1)$) и выбираем лучшую по AIC.

На её основе строим **иллюстративный прогноз** на 1–3 года вперёд.

In [ ]:
print("=" * 60)
print("КОРРЕЛЯЦИИ И ПРОСТЫЕ РЕГРЕССИИ НА ТЕМПАХ РОСТА")
print("Работаем не с уровнями, а с ростами, чтобы не ловить ложные связи")
print("=" * 60)

analysis_df = df_main.copy().sort_values('Год').reset_index(drop=True)
analysis_df['Рост лофт, %'] = analysis_df['Цена лофт'].pct_change(fill_method=None) * 100
analysis_df['Рост посвят, %'] = analysis_df['Цена посвят'].pct_change(fill_method=None) * 100
analysis_df['Инфляция РФ, %'] = analysis_df['ИПЦ общий'] - 100
analysis_df['Алко инфляция, %'] = analysis_df['ИПЦ алкоголь/табак'] - 100
analysis_df['Культура/досуг, %'] = analysis_df['ИПЦ культура/досуг'] - 100
analysis_df['Аренда жилья, %'] = analysis_df['ИПЦ аренда жилья'] - 100
analysis_df['Рост реальных доходов, %'] = analysis_df['Реальные доходы'] - 100

macro_cols = ['Инфляция РФ, %', 'Алко инфляция, %', 'Культура/досуг, %',
              'Аренда жилья, %', 'Ключевая ставка', 'Рост реальных доходов, %']

for price_col, price_label in [('Рост лофт, %', 'Лофты'), ('Рост посвят, %', 'Посвяты')]:
    print(f"\n{price_label}:")
    for macro in macro_cols:
        sub = analysis_df[[price_col, macro]].dropna()
        if len(sub) >= 3:
            r, p = stats.pearsonr(sub[price_col], sub[macro])
            sig = "✅ значима" if p < 0.05 else "❌ не значима"
            strength = "сильная" if abs(r) > 0.7 else ("умеренная" if abs(r) > 0.4 else "слабая")
            direction = "↑" if r > 0 else "↓"
            print(f"  {macro:<25} r = {r:+.3f} {direction} ({strength})  p = {p:.4f} {sig}")

    for macro in ['Инфляция РФ, %', 'Ключевая ставка']:
        sub = analysis_df[[price_col, macro]].dropna()
        if len(sub) >= 3:
            slope, intercept, r, p, _ = stats.linregress(sub[macro], sub[price_col])
            print(f"  Регрессия {price_col} ~ {macro}: β = {slope:+.3f}, R² = {r**2:.3f}, p = {p:.4f}")

corr_cols = ['Рост лофт, %', 'Рост посвят, %'] + macro_cols
corr_matrix = analysis_df[corr_cols].corr()

fig = px.imshow(corr_matrix, text_auto='.2f', color_continuous_scale='RdBu_r',
                zmin=-1, zmax=1, title='Матрица корреляций на темпах роста')
fig.update_layout(template='plotly_white', width=800, height=650)
fig.show()

КОРРЕЛЯЦИИ И ПРОСТЫЕ РЕГРЕССИИ НА ТЕМПАХ РОСТА
Работаем не с уровнями, а с ростами, чтобы не ловить ложные связи

Лофты:
  Инфляция РФ, %            r = -0.006 ↓ (слабая)  p = 0.9894 ❌ не значима
  Алко инфляция, %          r = +0.642 ↑ (умеренная)  p = 0.2430 ❌ не значима
  Культура/досуг, %         r = +0.748 ↑ (сильная)  p = 0.1461 ❌ не значима
  Аренда жилья, %           r = +0.741 ↑ (сильная)  p = 0.1519 ❌ не значима
  Ключевая ставка           r = +0.839 ↑ (сильная)  p = 0.0367 ✅ значима
  Рост реальных доходов, %  r = +0.705 ↑ (сильная)  p = 0.1838 ❌ не значима
  Регрессия Рост лофт, % ~ Инфляция РФ, %: β = -0.018, R² = 0.000, p = 0.9894
  Регрессия Рост лофт, % ~ Ключевая ставка: β = +1.344, R² = 0.704, p = 0.0367

Посвяты:
  Инфляция РФ, %            r = -0.465 ↓ (умеренная)  p = 0.2458 ❌ не значима
  Алко инфляция, %          r = -0.751 ↓ (сильная)  p = 0.0516 ❌ не значима
  Культура/досуг, %         r = -0.670 ↓ (умеренная)  p = 0.0999 ❌ не значима
  Аренда жилья, %         

In [ ]:
print("=" * 60)
print("ARIMA — ПРОГНОЗ НА 1 И 3 ГОДА")
print("Сравниваем несколько маленьких моделей и берём лучшую по AIC")
print("=" * 60)

candidate_orders = [(1, 1, 0), (0, 1, 1), (1, 1, 1)]

for col, label, color in [('Цена лофт', 'Лофты', 'fuchsia'), ('Цена посвят', 'Посвяты', 'teal')]:
    sub = df_main[['Год', col]].dropna()
    series = sub[col].values
    years = sub['Год'].values

    if len(series) < 5:
        print(f"\n⚠️ {label}: недостаточно данных для ARIMA (нужно минимум 5 точек)")
        continue

    best_model = None
    best_result = None
    best_order = None

    for order in candidate_orders:
        try:
            model = ARIMA(series, order=order)
            result = model.fit()
            if best_result is None or result.aic < best_result.aic:
                best_model = model
                best_result = result
                best_order = order
        except Exception:
            pass

    if best_result is None:
        print(f"\n⚠️ {label}: ни одна ARIMA-модель не сошлась")
        continue

    naive_mae = mean_absolute_error(series[1:], series[:-1])
    fitted_values = np.asarray(best_result.fittedvalues)
    fitted_values = fitted_values[-(len(series) - 1):]
    arima_mae = mean_absolute_error(series[1:], fitted_values)
    mape = np.mean(np.abs((series[1:] - fitted_values) / series[1:])) * 100

    forecast_res = best_result.get_forecast(steps=3)
    forecast = forecast_res.predicted_mean
    forecast_ci = forecast_res.conf_int(alpha=0.05)

    last_year = int(years.max())
    future_years = list(range(last_year + 1, last_year + 4))

    print(f"\n{label}:")
    print(f"  Лучшая ARIMA = {best_order}")
    print(f"  AIC = {best_result.aic:.1f}")
    print(f"  ARIMA MAE = {arima_mae:,.0f} ₽")
    print(f"  Наивный MAE = {naive_mae:,.0f} ₽")
    print(f"  MAPE = {mape:.1f}%")
    print(f"  📈 Прогноз на {last_year + 1}: {forecast[0]:,.0f} ₽")
    print(f"  📈 Прогноз на {last_year + 3}: {forecast[2]:,.0f} ₽")

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=list(years), y=list(series), mode='lines+markers',
        name=f'{label} (факт)',
        line=dict(color=color, width=3),
        marker=dict(size=10)
    ))
    fig.add_trace(go.Scatter(
        x=future_years, y=list(forecast), mode='lines+markers',
        name=f'ARIMA {best_order}',
        line=dict(color=color, dash='dash', width=3),
        marker=dict(size=12, symbol='star')
    ))
    fig.add_trace(go.Scatter(
        x=future_years + future_years[::-1],
        y=list(forecast_ci[:, 1]) + list(forecast_ci[:, 0])[::-1],
        fill='toself',
        fillcolor='rgba(128,128,128,0.2)',
        line=dict(color='rgba(0,0,0,0)'),
        name='95% ДИ'
    ))
    fig.add_vline(x=last_year + 0.5, line_dash='dot', line_color='gray',
                  annotation_text='→ прогноз', annotation_position='top right')
    fig.update_layout(
        title=f'{label}: лучший ARIMA-прогноз (MAPE={mape:.1f}%)',
        xaxis_title='Год', yaxis_title='Средняя цена, ₽',
        template='plotly_white', hovermode='x unified',
        xaxis=dict(tickmode='linear', dtick=1)
    )
    fig.show()

ARIMA — ПРОГНОЗ НА 1 И 3 ГОДА
Сравниваем несколько маленьких моделей и берём лучшую по AIC

Лофты:
  Лучшая ARIMA = (1, 1, 0)
  AIC = 89.8
  ARIMA MAE = 79 ₽
  Наивный MAE = 160 ₽
  MAPE = 6.0%
  📈 Прогноз на 2027: 2,192 ₽
  📈 Прогноз на 2029: 2,513 ₽



Посвяты:
  Лучшая ARIMA = (1, 1, 0)
  AIC = 116.6
  ARIMA MAE = 255 ₽
  Наивный MAE = 258 ₽
  MAPE = 6.8%
  📈 Прогноз на 2026: 3,699 ₽
  📈 Прогноз на 2028: 3,695 ₽


In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import plotly.graph_objects as go

# =========================
# 1. Общая база для реальных цен
# =========================
COMMON_BASE_YEAR = 2019
TREND_START_YEAR = 2022

# проверяем нужные таблицы
for name in ['loft_yearly', 'posvyat_yearly', 'df_ipc_yearly']:
    if name not in globals():
        raise NameError(f"Не найдена таблица {name}")

ipc = df_ipc_yearly[['Год', 'ИПЦ']].dropna().copy()
ipc['Год'] = ipc['Год'].astype(int)
ipc = ipc.sort_values('Год').reset_index(drop=True)

# строим накопленный индекс цен относительно общей базы
years = sorted(ipc['Год'].unique())
cum = {COMMON_BASE_YEAR: 1.0}

for y in range(COMMON_BASE_YEAR + 1, max(years) + 1):
    if y in ipc['Год'].values:
        cpi = float(ipc.loc[ipc['Год'] == y, 'ИПЦ'].iloc[0])
        cum[y] = cum[y - 1] * (cpi / 100)

for y in range(COMMON_BASE_YEAR - 1, min(years) - 1, -1):
    cpi_next = float(ipc.loc[ipc['Год'] == y + 1, 'ИПЦ'].iloc[0])
    cum[y] = cum[y + 1] / (cpi_next / 100)

def to_common_real(table, label):
    out = table[['Год', 'Средняя цена']].copy()
    out['Год'] = out['Год'].astype(int)
    out['cum_factor'] = out['Год'].map(cum)
    out['real_common'] = out['Средняя цена'] / out['cum_factor']
    out['Тип'] = label
    return out

loft_real_common = to_common_real(loft_yearly, 'Лофты')
posvyat_real_common = to_common_real(posvyat_yearly, 'Посвяты')

# =========================
# 2. Линейные тренды на последних годах
# =========================
loft_trend = loft_real_common[loft_real_common['Год'] >= TREND_START_YEAR].dropna()
posv_trend = posvyat_real_common[posvyat_real_common['Год'] >= TREND_START_YEAR].dropna()

if len(loft_trend) < 2 or len(posv_trend) < 2:
    raise ValueError("Слишком мало точек для оценки тренда")

slope_loft, intercept_loft, r_loft, p_loft, _ = stats.linregress(loft_trend['Год'], loft_trend['real_common'])
slope_posv, intercept_posv, r_posv, p_posv, _ = stats.linregress(posv_trend['Год'], posv_trend['real_common'])

# точка пересечения трендов
if np.isclose(slope_loft, slope_posv):
    cross_year = np.nan
    cross_price = np.nan
else:
    cross_year = (intercept_posv - intercept_loft) / (slope_loft - slope_posv)
    cross_price = intercept_loft + slope_loft * cross_year

print("Тренд лофтов:")
print(f"  slope = {slope_loft:.2f} руб./год, R² = {r_loft**2:.3f}, p = {p_loft:.4f}")

print("Тренд посвятов:")
print(f"  slope = {slope_posv:.2f} руб./год, R² = {r_posv**2:.3f}, p = {p_posv:.4f}")

if np.isnan(cross_year):
    print("Пересечение трендов не найдено: наклоны почти одинаковые.")
else:
    print(f"\nОценка года пересечения трендов: ~ {cross_year:.2f}")
    print(f"Оценка цены в точке пересечения: ~ {cross_price:,.0f} руб. в ценах {COMMON_BASE_YEAR} года")

# =========================
# 3. График
# =========================
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=loft_real_common['Год'], y=loft_real_common['real_common'],
    mode='lines+markers', name='Лофты (реальные, общая база)',
    line=dict(color='fuchsia', width=3), marker=dict(size=9)
))

fig.add_trace(go.Scatter(
    x=posvyat_real_common['Год'], y=posvyat_real_common['real_common'],
    mode='lines+markers', name='Посвяты (реальные, общая база)',
    line=dict(color='teal', width=3), marker=dict(size=9)
))

# линии тренда
x_future = np.arange(TREND_START_YEAR, max(loft_real_common['Год'].max(), posvyat_real_common['Год'].max()) + 5)

fig.add_trace(go.Scatter(
    x=x_future, y=intercept_loft + slope_loft * x_future,
    mode='lines', name=f'Тренд лофтов с {TREND_START_YEAR}',
    line=dict(color='fuchsia', dash='dash')
))

fig.add_trace(go.Scatter(
    x=x_future, y=intercept_posv + slope_posv * x_future,
    mode='lines', name=f'Тренд посвятов с {TREND_START_YEAR}',
    line=dict(color='teal', dash='dash')
))

if not np.isnan(cross_year):
    fig.add_vline(
        x=cross_year,
        line_dash='dot',
        line_color='gray',
        annotation_text=f'пересечение ~ {cross_year:.1f}',
        annotation_position='top right'
    )
    fig.add_trace(go.Scatter(
        x=[cross_year], y=[cross_price],
        mode='markers', name='Точка пересечения',
        marker=dict(color='black', size=11, symbol='x')
    ))

fig.update_layout(
    title=f'Реальные цены в общей базе ({COMMON_BASE_YEAR}) и оценка пересечения трендов',
    xaxis_title='Год',
    yaxis_title=f'Цена в ценах {COMMON_BASE_YEAR} года, ₽',
    template='plotly_white',
    hovermode='x unified',
    xaxis=dict(tickmode='linear', dtick=1)
)

fig.show()

Тренд лофтов:
  slope = 97.84 руб./год, R² = 0.897, p = 0.0145
Тренд посвятов:
  slope = -202.38 руб./год, R² = 0.907, p = 0.0479

Оценка года пересечения трендов: ~ 2029.51
Оценка цены в точке пересечения: ~ 1,527 руб. в ценах 2019 года
